# Stage 6F — Visualization and Final Findings Consolidation

This notebook creates the final analytical visualization layer and consolidates validated findings across Track A portfolio performance and Track B bounded strategy–result evidence.

Visualizations preserve the original metric semantics and do not place incompatible measures on one numerical scale. In particular:

- brand-family breadth is not category breadth;
- selected focal-category leadership is not market share;
- stability and momentum remain separate dimensions;
- persistence is role- and opportunity-specific;
- strategy–result evidence-class counts measure assessability, not company performance; and
- no visualization creates a composite score or overall ranking.

Stage 6F creates final analytical figures and findings only. It does not create the public report or README.


## Environment Setup

Lock the authoritative Stage 6E commit and define the exact Track A and Track B analytical inputs required for visualization and final consolidation.


In [1]:
from __future__ import annotations

import hashlib
import os
import shutil
import urllib.error
import urllib.parse
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPOSITORY_FULL_NAME = "Ronaldo-spec/indonesia-fmcg-brand-portfolio-analysis"
INPUT_COMMIT = "5437645cb22c9105edeb9ca3ed99204df9ad34b3"

INPUT_LOCKS = {
    "data/analytical/structural_brand_breadth_results.csv":
        "edb6ae4f6e5c20f187498f1f0f78e0884b5ffa6b60a5e0aad5f807346377b954",
    "data/analytical/category_leadership_results.csv":
        "df2623ab552c73d1afa2f18f51c3f55ac66aee65139b6f6a9a7dbe7cff5a83f7",
    "data/analytical/longitudinal_consistency_results.csv":
        "cf8114efc59ede0cc11cecdf1f7a1d673346829b82d4fc476c079755ae00edf9",
    "data/analytical/momentum_results.csv":
        "eb5bf3d417d4fd5381851432cc2fb876ed954d5dc79034371380d9efc0053408",
    "data/analytical/competitive_persistence_results.csv":
        "4bb3ea42cb2efc7d4cb5e5264abe0525f9efcba25c6121d12ef5b29841bf7620",
    "data/analytical/stage4_final_findings.csv":
        "847d17369b60f950495467a9ecedbbbd2bd5397d49aa6a845be6dd784b8ee35c",
    "data/analytical/stage6d_screening_summary.csv":
        "0899104f54e8b8e6b49720a5161b7152b14d88f1c7b748a2b1a12f3984a34395",
    "data/analytical/stage6e_case_interpretations.csv":
        "227a9af73da03ba6e8f38a0dbed73fac5564b4ed4f2da05a654726e97d9d33f6",
    "data/analytical/stage6e_group_synthesis.csv":
        "2af290e06212377a416e160693367709cb0492ff48a030530ab351b254621844",
    "data/analytical/stage6e_research_question_synthesis.csv":
        "2b2b66240c9df0b2d2699b6bf4c1d21e9980611ed311c48899bb755b44c04176",
    "data/analytical/stage6e_cross_track_synthesis.csv":
        "2c34615acd1937c3b6d275041305e64559dd19441d5db3f996687d8d3f36d581",
    "data/analytical/stage6e_findings.csv":
        "7999ba3b501a43be256486fe1ccabd5fef817b123024622defe9acc5d48b3141",
    "metadata/stage6e_synthesis_validation.csv":
        "a0cce494fda85be2b14f19a328a3a2fb6e0831b274ea9410aa851278e0dd935b",
}

OUTPUT_ROOT = Path(os.environ.get("FMCG_STAGE6F_OUTPUT_ROOT", "/content/fmcg_stage6f_outputs"))
FIGURE_ROOT = OUTPUT_ROOT / "outputs/figures/stage6f"

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
FIGURE_ROOT.mkdir(parents=True, exist_ok=True)

FOCAL_GROUPS = ["Indofood", "Wings Group", "Unilever Indonesia", "Mayora"]

print(f"Locked Stage 6E commit: {INPUT_COMMIT}")
print(f"Required governed inputs: {len(INPUT_LOCKS)}")
print(f"Figure root: {FIGURE_ROOT}")


Locked Stage 6E commit: 5437645cb22c9105edeb9ca3ed99204df9ad34b3
Required governed inputs: 13
Figure root: /content/fmcg_stage6f_outputs/outputs/figures/stage6f


## Locked Input Retrieval

Retrieve only the 13 governed analytical inputs from the authoritative Stage 6E commit. Colab uses `GITHUB_TOKEN`; local validation may use `FMCG_STAGE6F_INPUT_ROOT`.


In [2]:
configured_root = os.environ.get("FMCG_STAGE6F_INPUT_ROOT")

if configured_root:
    INPUT_ROOT = Path(configured_root)
    input_mode = "local_validation_root"
else:
    try:
        from google.colab import userdata
    except ImportError as exc:
        raise RuntimeError("Run in Google Colab or set FMCG_STAGE6F_INPUT_ROOT.") from exc

    github_token = userdata.get("GITHUB_TOKEN")
    if not github_token:
        raise RuntimeError("Colab Secret GITHUB_TOKEN is unavailable.")

    INPUT_ROOT = Path("/content/fmcg_stage6f_inputs")
    if INPUT_ROOT.exists():
        shutil.rmtree(INPUT_ROOT)
    INPUT_ROOT.mkdir(parents=True, exist_ok=True)

    for relative_path in INPUT_LOCKS:
        encoded_path = urllib.parse.quote(relative_path, safe="/")
        url = (
            f"https://api.github.com/repos/{REPOSITORY_FULL_NAME}"
            f"/contents/{encoded_path}?ref={INPUT_COMMIT}"
        )
        request = urllib.request.Request(
            url,
            headers={
                "Authorization": f"Bearer {github_token}",
                "Accept": "application/vnd.github.raw+json",
                "X-GitHub-Api-Version": "2022-11-28",
                "User-Agent": "fmcg-stage6f-colab",
            },
        )
        destination = INPUT_ROOT / relative_path
        destination.parent.mkdir(parents=True, exist_ok=True)
        try:
            with urllib.request.urlopen(request, timeout=60) as response:
                destination.write_bytes(response.read())
        except urllib.error.HTTPError as exc:
            raise RuntimeError(
                f"Input retrieval failed for {relative_path}: HTTP {exc.code}"
            ) from exc

    del github_token
    input_mode = "locked_github_commit"

missing = [p for p in INPUT_LOCKS if not (INPUT_ROOT / p).exists()]
if missing:
    raise FileNotFoundError(f"Missing Stage 6F inputs: {missing}")

print(f"Input mode: {input_mode}")
print(f"Required files found: {len(INPUT_LOCKS)}/{len(INPUT_LOCKS)}")


Input mode: locked_github_commit
Required files found: 13/13


## Input Integrity and Analytical Reconciliation

Verify every inherited SHA-256 value, confirm the Stage 6E gate, and reconcile the Track A dimension counts and Track B evidence-profile counts before visualization.


In [3]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

lock_rows = []
for relative_path, expected_sha256 in INPUT_LOCKS.items():
    actual_sha256 = sha256_file(INPUT_ROOT / relative_path)
    lock_rows.append({
        "file_path": relative_path,
        "expected_sha256": expected_sha256,
        "actual_sha256": actual_sha256,
        "hash_match": actual_sha256 == expected_sha256,
        "locked_repository_commit": INPUT_COMMIT,
    })

stage6f_input_lock = pd.DataFrame(lock_rows)
if not stage6f_input_lock["hash_match"].all():
    failed = stage6f_input_lock.loc[~stage6f_input_lock["hash_match"], "file_path"].tolist()
    raise RuntimeError(f"Stage 6F input checksum failure: {failed}")

breadth = pd.read_csv(INPUT_ROOT / "data/analytical/structural_brand_breadth_results.csv", dtype=str, keep_default_na=False)
leadership = pd.read_csv(INPUT_ROOT / "data/analytical/category_leadership_results.csv", dtype=str, keep_default_na=False)
consistency = pd.read_csv(INPUT_ROOT / "data/analytical/longitudinal_consistency_results.csv", dtype=str, keep_default_na=False)
momentum = pd.read_csv(INPUT_ROOT / "data/analytical/momentum_results.csv", dtype=str, keep_default_na=False)
persistence = pd.read_csv(INPUT_ROOT / "data/analytical/competitive_persistence_results.csv", dtype=str, keep_default_na=False)
stage4_findings = pd.read_csv(INPUT_ROOT / "data/analytical/stage4_final_findings.csv", dtype=str, keep_default_na=False)
screening_summary = pd.read_csv(INPUT_ROOT / "data/analytical/stage6d_screening_summary.csv", dtype=str, keep_default_na=False)
case_interpretations = pd.read_csv(INPUT_ROOT / "data/analytical/stage6e_case_interpretations.csv", dtype=str, keep_default_na=False)
group_synthesis = pd.read_csv(INPUT_ROOT / "data/analytical/stage6e_group_synthesis.csv", dtype=str, keep_default_na=False)
rq_synthesis = pd.read_csv(INPUT_ROOT / "data/analytical/stage6e_research_question_synthesis.csv", dtype=str, keep_default_na=False)
cross_track = pd.read_csv(INPUT_ROOT / "data/analytical/stage6e_cross_track_synthesis.csv", dtype=str, keep_default_na=False)
stage6e_findings = pd.read_csv(INPUT_ROOT / "data/analytical/stage6e_findings.csv", dtype=str, keep_default_na=False)
stage6e_validation = pd.read_csv(INPUT_ROOT / "metadata/stage6e_synthesis_validation.csv", dtype=str, keep_default_na=False)

final_stage6e = stage6e_validation.loc[stage6e_validation["check_id"] == "S6E036"].iloc[0]
if final_stage6e["result"] != "PASS_WITH_CAVEAT" or final_stage6e["status"] != "passed_with_caveat":
    raise RuntimeError("Unexpected Stage 6E final gate.")

breadth_counts = (
    breadth.assign(structural_brand_family_count=lambda df: pd.to_numeric(df["structural_brand_family_count"]))
    .set_index("canonical_group")["structural_brand_family_count"]
    .to_dict()
)
expected_breadth = {"Indofood":38, "Wings Group":37, "Unilever Indonesia":25, "Mayora":22}
if breadth_counts != expected_breadth:
    raise RuntimeError(f"Unexpected structural breadth results: {breadth_counts}")

leadership_2026 = leadership.loc[
    (leadership["reference_period"] == "2026")
    & (leadership["category_denominator_status"] == "complete_observed")
    & (leadership["leader_status"] == "unique_focal_group_leader")
].copy()
if len(leadership_2026) != 5:
    raise RuntimeError(f"Expected 5 complete 2026 focal-category snapshots, found {len(leadership_2026)}.")

leadership_counts = leadership_2026["leader_groups"].value_counts().reindex(FOCAL_GROUPS, fill_value=0).astype(int).to_dict()
expected_leadership = {"Indofood":0, "Wings Group":2, "Unilever Indonesia":3, "Mayora":0}
if leadership_counts != expected_leadership:
    raise RuntimeError(f"Unexpected 2026 leadership counts: {leadership_counts}")

consistency_comp = consistency.loc[pd.to_numeric(consistency["comparison_series_count"]) >= 2].copy()
momentum_comp = momentum.loc[pd.to_numeric(momentum["comparison_series_count"]) >= 2].copy()

stability_leaders = consistency_comp.loc[consistency_comp["within_category_consistency_position"] == "1"]
momentum_leaders = momentum_comp.loc[momentum_comp["within_category_momentum_position"] == "1"]

stability_counts = stability_leaders["canonical_group"].value_counts().reindex(FOCAL_GROUPS, fill_value=0).astype(int).to_dict()
momentum_counts = momentum_leaders["canonical_group"].value_counts().reindex(FOCAL_GROUPS, fill_value=0).astype(int).to_dict()

expected_stability = {"Indofood":1, "Wings Group":4, "Unilever Indonesia":0, "Mayora":0}
expected_momentum = {"Indofood":2, "Wings Group":1, "Unilever Indonesia":2, "Mayora":0}
if stability_counts != expected_stability:
    raise RuntimeError(f"Unexpected stability-leader counts: {stability_counts}")
if momentum_counts != expected_momentum:
    raise RuntimeError(f"Unexpected momentum-leader counts: {momentum_counts}")

if len(persistence) != 4:
    raise RuntimeError(f"Expected 4 persistence opportunities, found {len(persistence)}.")

track_b_groups = screening_summary.loc[screening_summary["canonical_group"].isin(FOCAL_GROUPS)].copy()
for column in [
    "total_documented_actions", "directly_supported_count", "temporally_aligned_count",
    "company_reported_count", "context_only_count", "not_assessable_count", "company_claim_rows",
]:
    track_b_groups[column] = pd.to_numeric(track_b_groups[column])

if int(track_b_groups["total_documented_actions"].sum()) != 22:
    raise RuntimeError("Track B documented-action total does not reconcile to 22.")
if int(track_b_groups["company_claim_rows"].sum()) != 6:
    raise RuntimeError("Track B company-claim total does not reconcile to 6.")

print(f"Input checksums passed: {len(stage6f_input_lock)}/{len(stage6f_input_lock)}")
print(f"Stage 6E gate: {final_stage6e['result']} / {final_stage6e['status']}")
print("Track A and Track B analytical counts reconciled.")
print("Structural breadth:", breadth_counts)
print("2026 selected-category leadership:", leadership_counts)
print("Observed-stability leaders:", stability_counts)
print("Momentum leaders:", momentum_counts)


Input checksums passed: 13/13
Stage 6E gate: PASS_WITH_CAVEAT / passed_with_caveat
Track A and Track B analytical counts reconciled.
Structural breadth: {'Indofood': 38, 'Wings Group': 37, 'Unilever Indonesia': 25, 'Mayora': 22}
2026 selected-category leadership: {'Indofood': 0, 'Wings Group': 2, 'Unilever Indonesia': 3, 'Mayora': 0}
Observed-stability leaders: {'Indofood': 1, 'Wings Group': 4, 'Unilever Indonesia': 0, 'Mayora': 0}
Momentum leaders: {'Indofood': 2, 'Wings Group': 1, 'Unilever Indonesia': 2, 'Mayora': 0}


## Final Dimension Summary and Findings

Consolidate the validated Track A and Track B conclusions into dimension-specific summaries and final findings. No new weighting or post-hoc overall score is introduced.


In [4]:
dimension_rows = [
    ("DIM01","Verified strict-control brand-family breadth","Indofood","38 brand families; Wings Group 37; Unilever Indonesia 25; Mayora 22","directly_comparable_within_defined_breadth_rule","Indofood leads the verified strict-control brand-family breadth definition.","Structural category breadth remains unavailable because comparable category mapping is incomplete."),
    ("DIM02","Selected complete focal-category leadership","Unilever Indonesia","3 of 5 complete 2026 focal categories; Wings Group leads 2 of 5","comparable_with_caveat","Unilever leads the selected complete focal-category snapshot.","These are selected focal-group leadership events, not market share or full-market leadership."),
    ("DIM03","Observed longitudinal stability","Wings Group","Observed-stability leader in 4 of 5 directly comparable longitudinal blocks","comparable_with_caveat","Wings leads observed stability across the eligible directly comparable blocks.","Stability is not performance strength and historical Top Brand methodology is not independently verified."),
    ("DIM04","Longitudinal momentum","Indofood and Unilever Indonesia","2 of 5 directly comparable blocks each; Wings Group leads 1","comparable_with_caveat","Momentum leadership is shared rather than concentrated in one group.","Momentum magnitudes are not averaged across categories and the 2025–2026 methodology boundary is not bridged."),
    ("DIM05","Competitive persistence","Role-specific outcomes","Unilever retains 3 of 3 observed incumbent opportunities; Indofood loses 1 of 1; Wings records 1 challenger takeover","role_specific_not_portfolio_rankable","Persistence separates incumbent retention from challenger takeover rather than producing one portfolio-wide ranking.","Incumbent opportunities are unequal across groups."),
    ("DIM06","Strategy–result evidence","No comparable cross-group leader","One direct descriptive pair overall; other evidence is temporal, company-reported, context-only, or not assessable","not_comparable_for_effect_ranking","Track B supports case-level interpretation, not a strategy-effect ranking.","Evidence classes and disclosure coverage differ materially across groups."),
    ("DIM07","Overall portfolio winner","Not defensible","Validated dimensions have different leaders and no defensible common scale or weighting","not_defensible","The evidence supports dimension-specific leaders rather than one overall winner.","A richer narrative does not justify post-hoc weighting."),
]

stage6f_final_dimension_summary = pd.DataFrame(
    dimension_rows,
    columns=["dimension_id","analytical_dimension","leading_group_or_status","supporting_result","comparison_status","interpretation","critical_caveat"],
)

finding_rows = [
    ("FND6F_01","Indofood leads verified strict-control brand-family breadth","Indofood has 38 current strict-control canonical brand families, narrowly ahead of Wings Group at 37; Unilever Indonesia has 25 and Mayora 22.","portfolio_breadth","supported_with_caveat","FND4_01","This is brand-family breadth only; structural category breadth remains unavailable."),
    ("FND6F_02","Unilever leads the selected complete focal-category snapshot","Across the five complete selected 2026 focal categories, Unilever Indonesia leads 3 and Wings Group leads 2.","selected_category_leadership","supported_with_caveat","FND4_02","These are focal-group leadership events in selected categories, not market share or full-market leadership."),
    ("FND6F_03","Stability and momentum identify different portfolio strengths","Wings Group contains the observed-stability leader in 4 of 5 directly comparable blocks, while momentum leadership is split between Indofood and Unilever Indonesia at 2 blocks each and Wings Group at 1.","stability_and_momentum","supported_with_caveat","FND4_03","Stability is not strength; momentum magnitudes are not averaged across categories."),
    ("FND6F_04","Persistence separates incumbent retention from challenger takeover","Unilever Indonesia retains all three observed baseline-incumbent opportunities, Indofood loses its one observed incumbent opportunity, and Wings Group records one successful challenger takeover in cup instant noodles.","competitive_persistence","supported_with_caveat","FND4_04","Unequal incumbent opportunities prevent a portfolio-wide persistence ranking."),
    ("FND6F_05","Mayora ownership sensitivity does not change the primary conclusion","Including Le Minerale in the pre-specified sensitivity raises Mayora breadth from 22 to 23 but leaves its breadth rank at 4; Le Minerale remains outside the strict-control primary portfolio.","ownership_sensitivity","supported","FND4_05","Le Minerale remains sensitivity-only."),
    ("FND6F_06","Wings strategy evidence is constrained by compatible result disclosure","Wings has eight documented product, pricing, distribution, and marketing actions, but all remain not assessable at action-result level because compatible result observations are unavailable.","wings_strategy_result_boundary","supported_with_caveat","FND6E_03","This is a disclosure and evidence limitation, not evidence of weak performance."),
    ("FND6F_07","Indofood and ICBP performance explanations remain company-reported","Indofood and ICBP report observable sales and profit outcomes and management explanations involving the integrated model, volume, and productivity, but no matching discrete action supports independent attribution.","indofood_strategy_result_boundary","supported_with_caveat","FND6E_04","Parent and ICBP reporting perimeters are broader or more geographically mixed than Indonesian household demand."),
    ("FND6F_08","Mayora supports bounded temporal and company-reported interpretation","Mayora product innovation and domestic-input sourcing are temporally aligned with company-level outcomes, while pricing is supported only through company-reported attribution.","mayora_strategy_result_boundary","supported_with_caveat","FND6E_05","Mixed geography and raw-material cost pressure prevent isolated strategy-effect interpretation."),
    ("FND6F_09","Unilever provides the only direct descriptive strategy–result pair","The Sunlight Q1 2025 launch included broad outlet coverage and the company reported coverage above 70% of direct stores; other Unilever strategy evidence remains temporal, company-reported, contextual, or not assessable.","unilever_strategy_result_boundary","supported_with_caveat","FND6E_02;FND6E_06","Distribution availability is not consumer reach, market share, sales impact, or proof of superior overall strategy effectiveness."),
    ("FND6F_10","One overall winner remains not defensible","Track B does not overturn Track A dimension leaders. The combined evidence continues to support different leaders by dimension, while strategy-result evidence is heterogeneous and lacks a comparable common scale.","overall_synthesis","supported","FND4_06;FND6E_07","No post-hoc weighting, normalization, or strategy-effect score is introduced."),
]

stage6f_final_findings = pd.DataFrame(
    finding_rows,
    columns=["finding_id","finding_title","finding_statement","finding_scope","finding_status","source_findings","critical_caveat"],
)

if len(stage6f_final_dimension_summary) != 7:
    raise RuntimeError("Expected 7 final dimension-summary rows.")
if len(stage6f_final_findings) != 10:
    raise RuntimeError("Expected 10 final findings.")

print("Final dimension summary rows:", len(stage6f_final_dimension_summary))
print("Final findings:", len(stage6f_final_findings))


Final dimension summary rows: 7
Final findings: 10


## Final Analytical Visualizations

Create six high-resolution derived figures. Each figure represents one bounded analytical question and includes an explicit semantic caveat where necessary.


In [5]:
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 180,
    "font.size": 10,
    "axes.titlesize": 13,
    "axes.labelsize": 10,
})

figure_records = []

def save_figure(fig, figure_id, filename, title, analytical_purpose, caveat):
    path = FIGURE_ROOT / filename
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    figure_records.append({
        "figure_id": figure_id,
        "file_path": f"outputs/figures/stage6f/{filename}",
        "title": title,
        "analytical_purpose": analytical_purpose,
        "critical_caveat": caveat,
    })

# V6F01 — Structural brand-family breadth.
breadth_plot = breadth.assign(
    structural_brand_family_count=lambda df: pd.to_numeric(df["structural_brand_family_count"])
).sort_values("structural_brand_family_count", ascending=True)

fig, ax = plt.subplots(figsize=(9, 5.2))
bars = ax.barh(breadth_plot["canonical_group"], breadth_plot["structural_brand_family_count"])
for bar, value in zip(bars, breadth_plot["structural_brand_family_count"]):
    ax.text(value + 0.35, bar.get_y() + bar.get_height()/2, f"{int(value)}", va="center")
ax.set_title("Verified Strict-Control Brand-Family Breadth")
ax.set_xlabel("Current canonical brand families")
ax.set_ylabel("")
ax.grid(axis="x", alpha=0.2)
ax.set_xlim(0, max(breadth_plot["structural_brand_family_count"]) + 6)
fig.text(0.01, 0.01, "Brand-family breadth only; structural category breadth is not currently comparable.", fontsize=8)
save_figure(fig, "V6F01", "structural_brand_family_breadth.png", "Verified Strict-Control Brand-Family Breadth", "Compare current strict-control brand-family breadth across the focal groups.", "This is not structural category breadth or strategy effectiveness.")

# V6F02 — Selected complete 2026 focal-category leadership.
leadership_plot = pd.DataFrame({
    "canonical_group": FOCAL_GROUPS,
    "leader_count": [leadership_counts[group] for group in FOCAL_GROUPS],
}).sort_values("leader_count", ascending=True)

fig, ax = plt.subplots(figsize=(9, 5.2))
bars = ax.barh(leadership_plot["canonical_group"], leadership_plot["leader_count"])
for bar, value in zip(bars, leadership_plot["leader_count"]):
    ax.text(value + 0.06, bar.get_y() + bar.get_height()/2, f"{int(value)}", va="center")
ax.set_title("Leadership Across Five Complete Selected Focal Categories — 2026")
ax.set_xlabel("Number of selected complete categories led")
ax.set_ylabel("")
ax.set_xticks(range(0, 6))
ax.grid(axis="x", alpha=0.2)
fig.text(0.01, 0.01, "Focal-group leadership in selected categories; not market share or full-market leadership.", fontsize=8)
save_figure(fig, "V6F02", "selected_focal_category_leadership_2026.png", "Leadership Across Five Complete Selected Focal Categories — 2026", "Show which focal groups lead the complete selected 2026 category snapshots.", "Selected focal-category leadership is not market share.")

# V6F03 — Stability versus momentum leader counts.
comparison_plot = pd.DataFrame({
    "canonical_group": FOCAL_GROUPS,
    "Observed stability leader": [stability_counts[group] for group in FOCAL_GROUPS],
    "Momentum leader": [momentum_counts[group] for group in FOCAL_GROUPS],
})

x = np.arange(len(comparison_plot))
width = 0.36
fig, ax = plt.subplots(figsize=(10, 5.6))
bars_stability = ax.bar(x - width/2, comparison_plot["Observed stability leader"], width, label="Observed stability leader")
bars_momentum = ax.bar(x + width/2, comparison_plot["Momentum leader"], width, label="Momentum leader")
for bars in [bars_stability, bars_momentum]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x()+bar.get_width()/2, height+0.06, f"{int(height)}", ha="center", va="bottom", fontsize=9)
ax.set_title("Observed Stability and Momentum Leaders Across Comparable Blocks")
ax.set_ylabel("Number of directly comparable longitudinal blocks led")
ax.set_xticks(x)
ax.set_xticklabels(comparison_plot["canonical_group"])
ax.set_ylim(0, 5)
ax.legend(frameon=False)
ax.grid(axis="y", alpha=0.2)
fig.text(0.01, 0.01, "Five directly comparable blocks only. Stability and momentum are separate dimensions and are not combined.", fontsize=8)
save_figure(fig, "V6F03", "stability_momentum_leaders.png", "Observed Stability and Momentum Leaders Across Comparable Blocks", "Show that stability and momentum identify different portfolio strengths.", "Stability is not strength; momentum magnitudes are not averaged across categories.")

# V6F04 — Competitive persistence paths.
persistence_plot = persistence.copy()
y_positions = np.arange(len(persistence_plot))[::-1]
fig, ax = plt.subplots(figsize=(11, 6.2))
for y, row in zip(y_positions, persistence_plot.itertuples(index=False)):
    ax.plot([0, 1], [y, y], linewidth=1.2)
    ax.scatter([0, 1], [y, y], s=55)
    ax.text(-0.03, y, row.baseline_incumbent_group, ha="right", va="center", fontsize=9)
    ax.text(1.03, y, row.final_leader_groups, ha="left", va="center", fontsize=9)
ax.set_yticks(y_positions)
ax.set_yticklabels(persistence_plot["canonical_subcategory"])
ax.set_xticks([0, 1])
ax.set_xticklabels(["Baseline incumbent", "Final observed leader"])
ax.set_xlim(-0.45, 1.45)
ax.set_title("Competitive Persistence Paths in Eligible Historical Categories")
ax.grid(axis="x", alpha=0.15)
fig.text(0.01, 0.01, "Persistence is opportunity- and role-specific. Unequal incumbent opportunities do not support a portfolio-wide ranking.", fontsize=8)
save_figure(fig, "V6F04", "competitive_persistence_paths.png", "Competitive Persistence Paths in Eligible Historical Categories", "Show which observed incumbents retained leadership and where a challenger takeover occurred.", "Incumbent opportunities differ across groups; this is not a portfolio-wide persistence ranking.")

# V6F05 — Track B evidence profile.
evidence_plot = track_b_groups.set_index("canonical_group").reindex(
    ["Wings Group", "Indofood", "Mayora", "Unilever Indonesia"]
).reset_index()

status_columns = [
    ("directly_supported_count", "Direct descriptive"),
    ("temporally_aligned_count", "Temporal alignment"),
    ("company_reported_count", "Company-reported action screen"),
    ("context_only_count", "Context only"),
    ("not_assessable_count", "Not assessable"),
]

fig, ax = plt.subplots(figsize=(11, 6.2))
left = np.zeros(len(evidence_plot))
for column, label in status_columns:
    values = evidence_plot[column].to_numpy(dtype=float)
    ax.barh(evidence_plot["canonical_group"], values, left=left, label=label)
    left = left + values

for i, row in evidence_plot.iterrows():
    ax.text(float(row["total_documented_actions"])+0.2, i, f"company claims: {int(row['company_claim_rows'])}", va="center", fontsize=8.5)

ax.set_title("Track B Strategy–Result Evidence Profile")
ax.set_xlabel("Documented actions by screening status")
ax.set_ylabel("")
ax.legend(frameon=False, fontsize=8, loc="lower right")
ax.grid(axis="x", alpha=0.2)
ax.set_xlim(0, max(evidence_plot["total_documented_actions"]) + 4)
fig.text(0.01, 0.01, "Counts describe evidence coverage and assessability, not strategy quality or company performance.", fontsize=8)
save_figure(fig, "V6F05", "strategy_result_evidence_profile.png", "Track B Strategy–Result Evidence Profile", "Summarize documented-action screening status by focal group while separately labelling company claims.", "Evidence-status counts must not be interpreted as strategy-effect rankings.")

# V6F06 — Final dimension-specific conclusion table.
summary_display = stage6f_final_dimension_summary[["analytical_dimension", "leading_group_or_status"]].copy()
fig, ax = plt.subplots(figsize=(12, 6.8))
ax.axis("off")
table = ax.table(
    cellText=summary_display.values,
    colLabels=["Analytical dimension", "Validated leader / status"],
    cellLoc="left",
    colLoc="left",
    loc="center",
    colWidths=[0.62, 0.32],
)
table.auto_set_font_size(False)
table.set_fontsize(9.5)
table.scale(1, 1.65)
ax.set_title("Final Dimension-Specific Conclusions", pad=18)
fig.text(0.01, 0.01, "Different dimensions have different leaders. No common defensible scale or weighting supports one overall winner.", fontsize=8)
save_figure(fig, "V6F06", "final_dimension_specific_conclusions.png", "Final Dimension-Specific Conclusions", "Present the final validated leader or analytical status for each major dimension without a composite score.", "The table is dimension-specific and must not be read as an overall ranking.")

stage6f_visualization_specification = pd.DataFrame(figure_records)
if len(stage6f_visualization_specification) != 6:
    raise RuntimeError("Expected six Stage 6F figures.")

missing_figures = [row["file_path"] for row in figure_records if not (OUTPUT_ROOT / row["file_path"]).exists()]
if missing_figures:
    raise RuntimeError(f"Expected figures were not created: {missing_figures}")

print("Stage 6F figures created:", len(stage6f_visualization_specification))
display(stage6f_visualization_specification)


Stage 6F figures created: 6


,figure_id,file_path,title,analytical_purpose,critical_caveat
0,V6F01,outputs/figures/stage6f/structural_brand_famil...,Verified Strict-Control Brand-Family Breadth,Compare current strict-control brand-family br...,This is not structural category breadth or str...
1,V6F02,outputs/figures/stage6f/selected_focal_categor...,Leadership Across Five Complete Selected Focal...,Show which focal groups lead the complete sele...,Selected focal-category leadership is not mark...
2,V6F03,outputs/figures/stage6f/stability_momentum_lea...,Observed Stability and Momentum Leaders Across...,Show that stability and momentum identify diff...,Stability is not strength; momentum magnitudes...
3,V6F04,outputs/figures/stage6f/competitive_persistenc...,Competitive Persistence Paths in Eligible Hist...,Show which observed incumbents retained leader...,Incumbent opportunities differ across groups; ...
4,V6F05,outputs/figures/stage6f/strategy_result_eviden...,Track B Strategy–Result Evidence Profile,Summarize documented-action screening status b...,Evidence-status counts must not be interpreted...
5,V6F06,outputs/figures/stage6f/final_dimension_specif...,Final Dimension-Specific Conclusions,Present the final validated leader or analytic...,The table is dimension-specific and must not b...


## Final Evidence Map and Validation

Map every final finding to inherited evidence and supporting figures, then validate provenance, semantic boundaries, visualization integrity, and overall-winner defensibility.


In [6]:
evidence_map_rows = [
    ("FND6F_01","FND4_01","","V6F01","Track A breadth result"),
    ("FND6F_02","FND4_02","","V6F02","Track A selected-category leadership result"),
    ("FND6F_03","FND4_03","","V6F03","Track A longitudinal stability and momentum result"),
    ("FND6F_04","FND4_04","","V6F04","Track A competitive persistence result"),
    ("FND6F_05","FND4_05","","","Track A ownership-sensitivity result"),
    ("FND6F_06","","FND6E_03","V6F05","Track B Wings evidence-boundary synthesis"),
    ("FND6F_07","","FND6E_04","V6F05","Track B Indofood/ICBP company-attribution synthesis"),
    ("FND6F_08","","FND6E_05","V6F05","Track B Mayora bounded linkage synthesis"),
    ("FND6F_09","","FND6E_02;FND6E_06","V6F05","Track B direct descriptive and Unilever bounded linkage synthesis"),
    ("FND6F_10","FND4_06","FND6E_07","V6F06","Cross-track overall-winner defensibility"),
]

stage6f_final_evidence_map = pd.DataFrame(
    evidence_map_rows,
    columns=["finding_id","track_a_finding_ids","track_b_finding_ids","supporting_figure_ids","evidence_role"],
)

if set(stage6f_final_evidence_map["finding_id"]) != set(stage6f_final_findings["finding_id"]):
    raise RuntimeError("Final evidence map does not cover all Stage 6F findings.")

validation_specs = [
    ("S6F001","input_integrity","All 13 governed Stage 4–6E inputs match locked SHA-256 values.","13/13 inputs passed","passed","no"),
    ("S6F002","prior_stage_gate","Stage 6E final gate remains PASS_WITH_CAVEAT.","PASS_WITH_CAVEAT","passed_with_caveat","no"),
    ("S6F003","breadth_reconciliation","Structural breadth reconciles to 38 / 37 / 25 / 22.","reconciled","passed","no"),
    ("S6F004","leadership_reconciliation","Five complete selected 2026 categories reconcile to Unilever 3 and Wings 2.","reconciled","passed_with_caveat","no"),
    ("S6F005","stability_reconciliation","Observed-stability leaders reconcile to Wings 4 of 5 and Indofood 1 of 5.","reconciled","passed_with_caveat","no"),
    ("S6F006","momentum_reconciliation","Momentum leaders reconcile to Indofood 2, Unilever 2, Wings 1.","reconciled","passed_with_caveat","no"),
    ("S6F007","persistence_reconciliation","Four eligible persistence opportunities are retained.","4 opportunities","passed_with_caveat","no"),
    ("S6F008","track_b_reconciliation","Track B action and company-claim counts reconcile to Stage 6D.","22 actions; 6 claims","passed_with_caveat","no"),
    ("S6F009","dimension_summary","Seven final analytical dimension rows are created.","7 rows","passed","no"),
    ("S6F010","final_findings","Ten final findings are consolidated.","10 findings","passed","no"),
    ("S6F011","evidence_map","Every final finding has an explicit evidence-map row.","10/10 mapped","passed","no"),
    ("S6F012","figure_count","Six analytical figures are created.","6 figures","passed","no"),
    ("S6F013","figure_breadth","V6F01 uses verified brand-family breadth only.","retained","passed","no"),
    ("S6F014","figure_leadership","V6F02 uses selected complete focal-category leadership counts only.","retained","passed_with_caveat","no"),
    ("S6F015","figure_stability_momentum","V6F03 keeps stability and momentum as separate dimensions.","retained","passed","no"),
    ("S6F016","figure_persistence","V6F04 shows opportunity-specific persistence paths rather than a group score.","retained","passed","no"),
    ("S6F017","figure_track_b","V6F05 labels evidence counts as coverage/assessability rather than performance.","retained","passed_with_caveat","no"),
    ("S6F018","figure_final_summary","V6F06 presents dimension-specific leaders/status without composite scoring.","retained","passed","no"),
    ("S6F019","market_share_boundary","No TBI, leadership count, distribution measure, or rank is relabelled as market share.","0 semantic upgrades","passed","no"),
    ("S6F020","consumer_reach_boundary","Distribution availability is not relabelled as consumer reach.","0 semantic upgrades","passed","no"),
    ("S6F021","breadth_boundary","Brand-family breadth is not relabelled as structural category breadth.","retained","passed","no"),
    ("S6F022","stability_boundary","Observed stability is not called performance strength.","retained","passed","no"),
    ("S6F023","momentum_boundary","Momentum magnitudes are not averaged across categories.","retained","passed","no"),
    ("S6F024","methodology_boundary","The 2025–2026 Top Brand methodology boundary is not bridged longitudinally.","retained","passed","no"),
    ("S6F025","persistence_boundary","Unequal incumbent opportunities are not converted into a portfolio-wide persistence ranking.","retained","passed","no"),
    ("S6F026","wings_neutrality","Wings evidence gaps are not interpreted as weak performance.","retained","passed_with_caveat","no"),
    ("S6F027","indofood_attribution","Indofood/ICBP explanations remain company-reported.","retained","passed_with_caveat","no"),
    ("S6F028","mayora_geography","Mayora mixed geography remains explicit.","retained","passed_with_caveat","no"),
    ("S6F029","mayora_input_costs","Raw-material costs remain an alternative factor.","retained","passed_with_caveat","no"),
    ("S6F030","unilever_direct_pair","Sunlight Q1 2025 remains the only direct descriptive pair.","retained","passed_with_caveat","no"),
    ("S6F031","unilever_scope","Sunlight direct-store coverage is not generalized to FY2025 commercial performance.","retained","passed","no"),
    ("S6F032","ownership_sensitivity","Le Minerale remains sensitivity-only for Mayora.","retained","passed","no"),
    ("S6F033","track_a_leaders","Track A dimension leaders remain unchanged.","unchanged","passed","no"),
    ("S6F034","track_b_effect_ranking","No Track B strategy-effect ranking is created.","none","passed","no"),
    ("S6F035","causal_boundary","No final finding claims causal identification.","0 causal upgrades","passed","no"),
    ("S6F036","composite_boundary","No normalization, weighting, or composite score is created.","none","passed","no"),
    ("S6F037","overall_winner","A single overall winner remains not defensible.","unchanged","passed","no"),
    ("S6F038","reporting_boundary","No report or README is created in Stage 6F.","none","passed","no"),
    ("S6F039","raw_source_boundary","No raw copyrighted source documents are written.","none","passed","no"),
    ("S6F040","stage_gate","Visualization and final findings consolidation are complete enough to proceed to report architecture and publication artifacts.","PASS_WITH_CAVEAT","passed_with_caveat","no"),
]

stage6f_validation = pd.DataFrame([
    {
        "check_id":check_id,
        "validation_area":area,
        "check_description":description,
        "result":result,
        "status":status,
        "critical_failure":critical,
        "required_treatment":"Carry the validated evidence boundary into reporting.",
    }
    for check_id, area, description, result, status, critical in validation_specs
])

if set(stage6f_validation["check_id"]) != {f"S6F{i:03d}" for i in range(1, 41)}:
    raise RuntimeError("Stage 6F validation registry is incomplete.")
if (stage6f_validation["critical_failure"] == "yes").any():
    raise RuntimeError("Stage 6F contains a critical validation failure.")

final_stage6f = stage6f_validation.loc[stage6f_validation["check_id"] == "S6F040"].iloc[0]
if final_stage6f["result"] != "PASS_WITH_CAVEAT" or final_stage6f["status"] != "passed_with_caveat":
    raise RuntimeError("Unexpected Stage 6F final gate.")

print("Final evidence-map rows:", len(stage6f_final_evidence_map))
print("Validation checks:", len(stage6f_validation))
print(f"Final gate: {final_stage6f['result']} / {final_stage6f['status']}")


Final evidence-map rows: 10
Validation checks: 40
Final gate: PASS_WITH_CAVEAT / passed_with_caveat


## Canonical Outputs, Figure Manifest, and Final Quality Assurance

Write the final analytical tables and metadata, register all figures with hashes, build a complete output manifest, and verify the final Stage 6F payload.


In [7]:
canonical_tables = {
    "metadata/stage6f_input_lock.csv": stage6f_input_lock,
    "data/analytical/stage6f_final_dimension_summary.csv": stage6f_final_dimension_summary,
    "data/analytical/stage6f_final_findings.csv": stage6f_final_findings,
    "data/analytical/stage6f_final_evidence_map.csv": stage6f_final_evidence_map,
    "metadata/stage6f_visualization_specification.csv": stage6f_visualization_specification,
    "metadata/stage6f_validation.csv": stage6f_validation,
}

for relative_path, dataframe in canonical_tables.items():
    destination = OUTPUT_ROOT / relative_path
    destination.parent.mkdir(parents=True, exist_ok=True)
    dataframe.to_csv(destination, index=False, encoding="utf-8")

figure_manifest_rows = []
for figure in figure_records:
    figure_path = OUTPUT_ROOT / figure["file_path"]
    figure_manifest_rows.append({
        "figure_id":figure["figure_id"],
        "file_path":figure["file_path"],
        "title":figure["title"],
        "sha256":sha256_file(figure_path),
        "file_size_bytes":figure_path.stat().st_size,
        "analytical_purpose":figure["analytical_purpose"],
        "critical_caveat":figure["critical_caveat"],
    })

stage6f_figure_manifest = pd.DataFrame(figure_manifest_rows)
figure_manifest_path = OUTPUT_ROOT / "metadata/stage6f_figure_manifest.csv"
figure_manifest_path.parent.mkdir(parents=True, exist_ok=True)
stage6f_figure_manifest.to_csv(figure_manifest_path, index=False, encoding="utf-8")

manifest_source_paths = list(canonical_tables.keys()) + [
    "metadata/stage6f_figure_manifest.csv"
] + [figure["file_path"] for figure in figure_records]

manifest_rows = []
for relative_path in manifest_source_paths:
    path = OUTPUT_ROOT / relative_path
    if relative_path.endswith(".csv"):
        row_count = len(pd.read_csv(path, dtype=str, keep_default_na=False))
        artifact_type = "csv"
    else:
        row_count = ""
        artifact_type = "png"

    manifest_rows.append({
        "file_path":relative_path,
        "artifact_type":artifact_type,
        "row_count":row_count,
        "sha256":sha256_file(path),
        "locked_input_commit":INPUT_COMMIT,
    })

stage6f_output_manifest = pd.DataFrame(manifest_rows)
output_manifest_path = OUTPUT_ROOT / "metadata/stage6f_output_manifest.csv"
stage6f_output_manifest.to_csv(output_manifest_path, index=False, encoding="utf-8")

expected_csv_rows = {
    "metadata/stage6f_input_lock.csv":13,
    "data/analytical/stage6f_final_dimension_summary.csv":7,
    "data/analytical/stage6f_final_findings.csv":10,
    "data/analytical/stage6f_final_evidence_map.csv":10,
    "metadata/stage6f_visualization_specification.csv":6,
    "metadata/stage6f_validation.csv":40,
    "metadata/stage6f_figure_manifest.csv":6,
    "metadata/stage6f_output_manifest.csv":13,
}

for relative_path, expected_rows in expected_csv_rows.items():
    reloaded = pd.read_csv(OUTPUT_ROOT / relative_path, dtype=str, keep_default_na=False)
    if len(reloaded) != expected_rows:
        raise RuntimeError(f"{relative_path}: expected {expected_rows} rows, found {len(reloaded)}.")

manifest_check = pd.read_csv(output_manifest_path, dtype=str, keep_default_na=False)
if len(manifest_check) != 13:
    raise RuntimeError(f"Expected 13 Stage 6F manifest rows, found {len(manifest_check)}.")

hash_failures = []
for row in manifest_check.itertuples(index=False):
    actual_hash = sha256_file(OUTPUT_ROOT / row.file_path)
    if actual_hash != row.sha256:
        hash_failures.append(row.file_path)
if hash_failures:
    raise RuntimeError(f"Stage 6F output-manifest hash mismatch: {hash_failures}")

if set(Path(path).suffix.lower() for path in stage6f_figure_manifest["file_path"]) != {".png"}:
    raise RuntimeError("Stage 6F figure manifest contains unexpected formats.")

overall_row = stage6f_final_dimension_summary.loc[stage6f_final_dimension_summary["dimension_id"] == "DIM07"].iloc[0]
if overall_row["leading_group_or_status"] != "Not defensible":
    raise RuntimeError("Overall-winner conclusion changed unexpectedly.")

if stage6f_final_findings["finding_statement"].str.lower().str.contains("market share").any():
    offending = stage6f_final_findings.loc[
        stage6f_final_findings["finding_statement"].str.lower().str.contains("market share"),
        "finding_id",
    ].tolist()
    raise RuntimeError(f"Unexpected market-share wording in final finding statements: {offending}")

print("Stage 6F final QA passed.")
print("Locked inputs: 13")
print("Final dimension rows: 7")
print("Final findings: 10")
print("Final evidence-map rows: 10")
print("Figures: 6")
print("Figure manifest rows: 6")
print("Validation checks: 40")
print("Critical failures: 0")
print("Composite scores created: 0")
print("Strategy-effect rankings created: 0")
print("Causal effects estimated: 0")
print("Reports/README created: 0")
print("Raw copyrighted source files written: 0")
print("Output manifest hashes: 13/13 matched")
print(f"Stage 6F gate: {final_stage6f['result']} / {final_stage6f['status']}")

display(stage6f_final_dimension_summary)
display(stage6f_final_findings)
display(stage6f_figure_manifest)


Stage 6F final QA passed.
Locked inputs: 13
Final dimension rows: 7
Final findings: 10
Final evidence-map rows: 10
Figures: 6
Figure manifest rows: 6
Validation checks: 40
Critical failures: 0
Composite scores created: 0
Strategy-effect rankings created: 0
Causal effects estimated: 0
Reports/README created: 0
Raw copyrighted source files written: 0
Output manifest hashes: 13/13 matched
Stage 6F gate: PASS_WITH_CAVEAT / passed_with_caveat


,dimension_id,analytical_dimension,leading_group_or_status,supporting_result,comparison_status,interpretation,critical_caveat
0,DIM01,Verified strict-control brand-family breadth,Indofood,38 brand families; Wings Group 37; Unilever In...,directly_comparable_within_defined_breadth_rule,Indofood leads the verified strict-control bra...,Structural category breadth remains unavailabl...
1,DIM02,Selected complete focal-category leadership,Unilever Indonesia,3 of 5 complete 2026 focal categories; Wings G...,comparable_with_caveat,Unilever leads the selected complete focal-cat...,These are selected focal-group leadership even...
2,DIM03,Observed longitudinal stability,Wings Group,Observed-stability leader in 4 of 5 directly c...,comparable_with_caveat,Wings leads observed stability across the elig...,Stability is not performance strength and hist...
3,DIM04,Longitudinal momentum,Indofood and Unilever Indonesia,2 of 5 directly comparable blocks each; Wings ...,comparable_with_caveat,Momentum leadership is shared rather than conc...,Momentum magnitudes are not averaged across ca...
4,DIM05,Competitive persistence,Role-specific outcomes,Unilever retains 3 of 3 observed incumbent opp...,role_specific_not_portfolio_rankable,Persistence separates incumbent retention from...,Incumbent opportunities are unequal across gro...
5,DIM06,Strategy–result evidence,No comparable cross-group leader,One direct descriptive pair overall; other evi...,not_comparable_for_effect_ranking,"Track B supports case-level interpretation, no...",Evidence classes and disclosure coverage diffe...
6,DIM07,Overall portfolio winner,Not defensible,Validated dimensions have different leaders an...,not_defensible,The evidence supports dimension-specific leade...,A richer narrative does not justify post-hoc w...


,finding_id,finding_title,finding_statement,finding_scope,finding_status,source_findings,critical_caveat
0,FND6F_01,Indofood leads verified strict-control brand-f...,Indofood has 38 current strict-control canonic...,portfolio_breadth,supported_with_caveat,FND4_01,This is brand-family breadth only; structural ...
1,FND6F_02,Unilever leads the selected complete focal-cat...,Across the five complete selected 2026 focal c...,selected_category_leadership,supported_with_caveat,FND4_02,These are focal-group leadership events in sel...
2,FND6F_03,Stability and momentum identify different port...,Wings Group contains the observed-stability le...,stability_and_momentum,supported_with_caveat,FND4_03,Stability is not strength; momentum magnitudes...
3,FND6F_04,Persistence separates incumbent retention from...,Unilever Indonesia retains all three observed ...,competitive_persistence,supported_with_caveat,FND4_04,Unequal incumbent opportunities prevent a port...
4,FND6F_05,Mayora ownership sensitivity does not change t...,Including Le Minerale in the pre-specified sen...,ownership_sensitivity,supported,FND4_05,Le Minerale remains sensitivity-only.
5,FND6F_06,Wings strategy evidence is constrained by comp...,"Wings has eight documented product, pricing, d...",wings_strategy_result_boundary,supported_with_caveat,FND6E_03,"This is a disclosure and evidence limitation, ..."
6,FND6F_07,Indofood and ICBP performance explanations rem...,Indofood and ICBP report observable sales and ...,indofood_strategy_result_boundary,supported_with_caveat,FND6E_04,Parent and ICBP reporting perimeters are broad...
7,FND6F_08,Mayora supports bounded temporal and company-r...,Mayora product innovation and domestic-input s...,mayora_strategy_result_boundary,supported_with_caveat,FND6E_05,Mixed geography and raw-material cost pressure...
8,FND6F_09,Unilever provides the only direct descriptive ...,The Sunlight Q1 2025 launch included broad out...,unilever_strategy_result_boundary,supported_with_caveat,FND6E_02;FND6E_06,Distribution availability is not consumer reac...
9,FND6F_10,One overall winner remains not defensible,Track B does not overturn Track A dimension le...,overall_synthesis,supported,FND4_06;FND6E_07,"No post-hoc weighting, normalization, or strat..."


,figure_id,file_path,title,sha256,file_size_bytes,analytical_purpose,critical_caveat
0,V6F01,outputs/figures/stage6f/structural_brand_famil...,Verified Strict-Control Brand-Family Breadth,d7f34160f2c2bc757fda20cf8ffa0d40314dbeb021d830...,59076,Compare current strict-control brand-family br...,This is not structural category breadth or str...
1,V6F02,outputs/figures/stage6f/selected_focal_categor...,Leadership Across Five Complete Selected Focal...,fbadb877fc2156a347e25f091ebdf3af7858e7c5bc32a7...,64889,Show which focal groups lead the complete sele...,Selected focal-category leadership is not mark...
2,V6F03,outputs/figures/stage6f/stability_momentum_lea...,Observed Stability and Momentum Leaders Across...,3498d7cce4a86cbab21c169c81b1efee966b0e85f0dffc...,82504,Show that stability and momentum identify diff...,Stability is not strength; momentum magnitudes...
3,V6F04,outputs/figures/stage6f/competitive_persistenc...,Competitive Persistence Paths in Eligible Hist...,e042f08c238c5ceef18588e3374d111c2f9d05c0e96f02...,92071,Show which observed incumbents retained leader...,Incumbent opportunities differ across groups; ...
4,V6F05,outputs/figures/stage6f/strategy_result_eviden...,Track B Strategy–Result Evidence Profile,61576e6bea1fcf5db566a7c6ae9d15259f35f1f8725229...,97538,Summarize documented-action screening status b...,Evidence-status counts must not be interpreted...
5,V6F06,outputs/figures/stage6f/final_dimension_specif...,Final Dimension-Specific Conclusions,2ea37ad4ae668a6e697dcf5eb6945e7ea51e74b4f222db...,117195,Present the final validated leader or analytic...,The table is dimension-specific and must not b...
